In [0]:
# Gold layer implementation

from pyspark.sql import functions as F

sales = spark.table(
    "retail_catalog.silver.sales_clean"
)

In [0]:
sales_summary = sales.agg(

    F.countDistinct("SalesID")
        .alias("TotalTransactions"),

    F.countDistinct("CustomerID")
        .alias("TotalCustomers"),

    F.sum("Quantity")
        .alias("TotalUnitsSold"),

    F.sum("Revenue")
        .alias("TotalRevenue"),

    F.round(F.avg("Revenue"), 2)
        .alias("AverageTransactionValue")
)

display(sales_summary)

In [0]:
category_sales = sales.groupBy(
    "Category"
).agg(

    F.countDistinct("SalesID")
        .alias("TotalTransactions"),

    F.sum("Quantity")
        .alias("TotalQuantity"),

    F.sum("Revenue")
        .alias("TotalRevenue")

).orderBy(
    F.desc("TotalRevenue")
)

display(category_sales)

In [0]:
city_sales = sales.groupBy(
    "City"
).agg(

    F.countDistinct("CustomerID")
        .alias("TotalCustomers"),

    F.countDistinct("SalesID")
        .alias("TotalTransactions"),

    F.sum("Revenue")
        .alias("TotalRevenue")

).orderBy(
    F.desc("TotalRevenue")
)

display(city_sales)

In [0]:
monthly_sales = sales.withColumn(
    "Month",
    F.date_format("SalesDate", "yyyy-MM")
).groupBy(
    "Month"
).agg(

    F.countDistinct("SalesID")
        .alias("TotalTransactions"),

    F.sum("Quantity")
        .alias("TotalQuantity"),

    F.sum("Revenue")
        .alias("TotalRevenue")

).orderBy("Month")

display(monthly_sales)

In [0]:
customer_sales = sales.groupBy(
    "CustomerID",
    "CustomerName",
    "City"
).agg(

    F.countDistinct("SalesID")
        .alias("TotalTransactions"),

    F.sum("Revenue")
        .alias("TotalRevenue")

).withColumn(

    "CustomerSegment",

    F.when(
        F.col("TotalRevenue") >= 75000,
        "Premium"
    ).when(
        F.col("TotalRevenue") >= 30000,
        "Regular"
    ).otherwise(
        "Standard"
    )

).orderBy(
    F.desc("TotalRevenue")
)

display(customer_sales)

In [0]:
product_sales = sales.groupBy(
    "ProductID",
    "ProductName",
    "Category"
).agg(

    F.sum("Quantity")
        .alias("TotalQuantity"),

    F.sum("Revenue")
        .alias("TotalRevenue")

).orderBy(
    F.desc("TotalRevenue")
)

display(product_sales)

In [0]:
gold_tables = {
    "sales_summary": sales_summary,
    "category_sales": category_sales,
    "city_sales": city_sales,
    "monthly_sales": monthly_sales,
    "customer_sales": customer_sales,
    "product_sales": product_sales
}

for table_name, dataframe in gold_tables.items():

    dataframe.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(
            f"retail_catalog.gold.{table_name}"
        )

    print(f"Gold table created: {table_name}")

In [0]:
%sql
SHOW TABLES IN retail_catalog.gold;

In [0]:
%sql
SELECT *
FROM retail_catalog.gold.sales_summary;